# 🤖 NLP: Когда машины начинают понимать нас
### Практикум · Google Colab

**Выполнила:** Манина В. С., группа ББИ2302, МТУСИ

| Часть | Тема |
|-------|------|
| [Часть 1](#part1) | NLTK — токенизация, стемминг, лемматизация, стоп-слова |
| [Часть 2](#part2) | spaCy — POS-теггинг, NER, синтаксический анализ |
| [Часть 3](#part3) | Natasha — морфология и NER для русского языка |
| [Часть 4](#part4) | RegEx — гибридный извлекатель (NER + RegEx) |
| [Часть 5](#part5) | Word2Vec, FastText, Gensim — векторные слова |
| [Часть 6](#part6) | Нейросети для NLP (Keras SimpleRNN + PyTorch LSTM) |
| [Часть 7](#part7) | Классификация текста (scikit-learn + Lenta.ru) |
| [Часть 8](#part8) | BERT — трансформер для анализа тональности |

---
> 💡 Для GPU: `Среда выполнения → Изменить тип → GPU`  
> ⚠️ Некоторые ячейки долго загружают модели — это нормально.

## 🔧 Установка всех библиотек

In [ ]:
# Устанавливаем всё необходимое
!pip install nltk pymorphy3 -q
!pip install setuptools==69.5.1 -q
!pip install natasha -q
!pip install spacy -q
!python -m spacy download ru_core_news_lg -q
!pip install gensim -q
!pip install navec wget -q
!pip install torch -q
!pip install transformers datasets -q
!pip install scikit-learn kagglehub[pandas-datasets] -q
!pip install seaborn -q
print("✅ Всё установлено!")

In [ ]:
# Текст для большинства задач (Habr)
TEXT = """«Хабр» (ранее «Хабрахабр») — это крупнейшее в Европе русскоязычное IT-сообщество,
основанное 26 мая 2006 года предпринимателем Денисом Крючковым, который в 2013 году
выкупил доли у Mail.Ru Group для единоличного управления проектом. С февраля 2020
года платформа сменила юрисдикцию с российской на международную, и в настоящее время
её владельцем является кипрская компания Habr Blockchain Publishing LTD,
зарегистрированная в Никосии, хотя в Москве продолжает функционировать связанное
юридическое лицо ООО «Хабр». Площадка активно развивает профессиональные мероприятия:
например, на 25 февраля 2026 года в Москве запланирован «Зимний тест-драйв Хабра
для компаний», а на 3–5 июля 2026 года — летняя антиконференция Summer Merge
в селе Успенское. ИНН организации: 7712345678."""
print("Текст загружен ✓")

---
<a id='part1'></a>
# Часть 1. NLTK — базовая предобработка текста
## Токенизация → Стемминг → Лемматизация → Стоп-слова

In [ ]:
import nltk
import pymorphy3

# Загрузка ресурсов NLTK
for pkg in ['punkt_tab', 'stopwords', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger']:
    nltk.download(pkg, quiet=True)

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem.snowball import SnowballStemmer
from nltk.corpus import stopwords

# 1. Токенизация
sentences = sent_tokenize(TEXT, language='russian')
words = word_tokenize(TEXT, language='russian')
print(f"Предложений: {len(sentences)}")
print(f"Первое предложение:\n  {sentences[0][:120]}...\n")
print(f"Первые 15 токенов: {words[:15]}")

In [ ]:
# 2. Стемминг — грубое отсечение окончаний
stemmer = SnowballStemmer("russian")
stemmed = [stemmer.stem(w) for w in words[:20]]
print("Стемминг (первые 20 слов):")
for orig, stem in zip(words[:20], stemmed):
    if orig != stem:
        print(f"  {orig:20s} → {stem}")

In [ ]:
# 3. Лемматизация — точное приведение к нормальной форме (pymorphy3)
morph = pymorphy3.MorphAnalyzer()
lemmatized = [morph.parse(w)[0].normal_form for w in words]
print("Лемматизация (первые 20):")
for orig, lem in zip(words[:20], lemmatized[:20]):
    if orig.lower() != lem:
        print(f"  {orig:20s} → {lem}")

In [ ]:
# 4. Удаление стоп-слов
stop_words = set(stopwords.words('russian'))
filtered = [w for w in lemmatized if w.lower() not in stop_words and w.isalpha()]
print(f"До удаления:  {len(lemmatized)} токенов")
print(f"После:        {len(filtered)} токенов")
print(f"\nОчищенные слова: {filtered[:20]}")

In [ ]:
# 5. Готовая функция предобработки
def preprocess_text(text):
    """Токенизация → Лемматизация → Удаление стоп-слов и пунктуации"""
    words = word_tokenize(text, language='russian')
    lemmas = [morph.parse(w)[0].normal_form for w in words]
    clean = [w for w in lemmas if w.lower() not in stop_words and w.isalpha()]
    return clean

result = preprocess_text(TEXT)
print(f"Итоговый pipeline: {len(result)} значимых слов")
print(result[:25])

In [ ]:
# 6. Частотный анализ
import matplotlib.pyplot as plt
from collections import Counter

freq = Counter(result)
top15 = freq.most_common(15)

words_plot, counts_plot = zip(*top15)
plt.figure(figsize=(12, 4))
plt.bar(words_plot, counts_plot, color='#3498db', edgecolor='black')
plt.title('Топ-15 значимых слов в тексте', fontsize=14)
plt.xlabel('Слово'); plt.ylabel('Частота')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# 7. Биграммы
from nltk import bigrams
clean_words = [w for w in word_tokenize(TEXT, language='russian') if w.isalpha()]
bi = list(bigrams(clean_words))
print("Биграммы (первые 10):")
for pair in bi[:10]:
    print(f"  {pair}")

### 📝 Самостоятельная работа 1

In [ ]:
# Задание 2: Стемминг vs Лемматизация — сравнительная таблица
import pandas as pd

text_beg = "Бегущий человек быстро бежал по беговой дорожке. Вокруг него бегали маленькие дети, устраивая весёлую беготню."
words_beg = word_tokenize(text_beg, language='russian')

rows = []
for w in words_beg:
    if w.isalpha():
        stem = stemmer.stem(w)
        lemma = morph.parse(w)[0].normal_form
        rows.append({'Слово': w, 'Стемминг': stem, 'Лемматизация': lemma})

df = pd.DataFrame(rows)
print("Сравнение: Стемминг vs Лемматизация")
print(df.to_string(index=False))

In [ ]:
# Задание 5: Марковский генератор текста
import random

fairy_tale = """Жил-был старый король в далеком замке. У короля была золотая корона и верный конь.
Конь любил скакать по широкому полю, а король любил смотреть на высокие горы.
В горах жили мудрые драконы, которые охраняли древние сокровища.
Сокровища эти были спрятаны в глубокой пещере, где всегда горел волшебный огонь."""

tokens = word_tokenize(fairy_tale, language='russian')
tokens = [t for t in tokens if t.isalpha()]

# Строим словарь: слово → список следующих слов
chain = {}
for i in range(len(tokens) - 1):
    key = tokens[i]
    nxt = tokens[i + 1]
    chain.setdefault(key, []).append(nxt)

# Генерируем фразу
start = random.choice(['король', 'конь', 'горах'])
phrase = [start]
for _ in range(6):
    word = phrase[-1]
    if word in chain:
        phrase.append(random.choice(chain[word]))
    else:
        break

print(f"Стартовое слово: '{start}'")
print(f"Сгенерированная фраза: {' '.join(phrase)}")

---
<a id='part2'></a>
# Часть 2. spaCy — промышленный NLP
## POS-теггинг, синтаксический анализ, NER, семантическое сходство

In [ ]:
import spacy
nlp = spacy.load('ru_core_news_lg')
doc = nlp(TEXT)
print(f"Токенов: {len(doc)}, Предложений: {len(list(doc.sents))}")

In [ ]:
# POS-теггинг: слово | лемма | часть речи
print(f"{'Слово':<22} {'Лемма':<22} {'POS':<8} Стоп?")
print("-" * 65)
for token in list(doc)[:25]:
    if not token.is_punct:
        stop = '✓' if token.is_stop else ''
        print(f"{token.text:<22} {token.lemma_:<22} {token.pos_:<8} {stop}")

In [ ]:
# Глаголы в начальной форме
verbs = [(t.text, t.lemma_) for t in doc if t.pos_ == 'VERB']
print("Глаголы в тексте:")
for text, lemma in verbs:
    print(f"  {text:20s} → {lemma}")

In [ ]:
# Синтаксическое дерево — визуализация
from spacy import displacy
first_sent = list(doc.sents)[0]
displacy.render(first_sent, style='dep', jupyter=True,
                options={'compact': True, 'distance': 90})

In [ ]:
# Извлечение именованных сущностей (NER)
print("Именованные сущности (spaCy):")
print(f"{'Текст':<35} {'Тип':<6} Описание")
print("-" * 75)
for ent in doc.ents:
    desc = spacy.explain(ent.label_) or ''
    print(f"{ent.text:<35} {ent.label_:<6} {desc}")

In [ ]:
# Визуализация NER прямо в ноутбуке
displacy.render(doc, style='ent', jupyter=True)

In [ ]:
# Семантическое сходство предложений
s1 = nlp("Я люблю кушать вкусные яблоки.")
s2 = nlp("Мне нравится есть сочные фрукты.")
s3 = nlp("Завтра будет облачная погода и сильный дождь.")

print("Семантическое сходство (cosine similarity):")
print(f"  s1 ↔ s2 (яблоки/фрукты): {s1.similarity(s2):.3f}")
print(f"  s1 ↔ s3 (яблоки/погода): {s1.similarity(s3):.3f}")
print(f"  s2 ↔ s3 (фрукты/погода): {s2.similarity(s3):.3f}")

In [ ]:
# Анонимизация — замена имён людей на [ДАННЫЕ УДАЛЕНЫ]
def anonymize(text, nlp_model):
    doc_tmp = nlp_model(text)
    result = text
    for ent in reversed(doc_tmp.ents):  # reversed — чтобы не сбивать индексы
        if ent.label_ == 'PER':
            result = result[:ent.start_char] + '[ДАННЫЕ УДАЛЕНЫ]' + result[ent.end_char:]
    return result

test_text = "На встрече присутствовали Иван Иванов и Анна Петрова. Они обсуждали проект с Алексеем."
print("До:",  test_text)
print("После:", anonymize(test_text, nlp))

---
<a id='part3'></a>
# Часть 3. Natasha — специалист по русскому языку
## POS-теггинг, NER, метрики качества (Precision/Recall/F1)

In [ ]:
from natasha import Doc, MorphVocab, NewsEmbedding, NewsMorphTagger, Segmenter, NewsNERTagger

# Инициализация компонентов
segmenter    = Segmenter()
embedding    = NewsEmbedding()
morph_tagger = NewsMorphTagger(embedding)
morph_vocab  = MorphVocab()
ner_tagger   = NewsNERTagger(embedding)

print("Natasha инициализирована ✓")

In [ ]:
# POS-теггирование и лемматизация
doc_n = Doc(TEXT)
doc_n.segment(segmenter)
doc_n.tag_morph(morph_tagger)

print(f"{'Слово':<22} {'Лемма':<22} {'POS'}")
print("-" * 55)
for token in list(doc_n.tokens)[:25]:
    token.lemmatize(morph_vocab)
    if token.pos and token.pos not in ['PUNCT', 'NUM']:
        print(f"{token.text:<22} {str(token.lemma):<22} {token.pos}")

In [ ]:
# NER — извлечение именованных сущностей
doc_n.tag_ner(ner_tagger)

print(f"{'Сущность':<35} {'Тип'}")
print("-" * 50)
for span in doc_n.spans:
    print(f"{span.text:<35} {span.type}")

In [ ]:
# Метрики качества NER (Precision / Recall / F1)
# Нормализуем для честного сравнения
def normalize(text):
    """Приводим к нижнему регистру для нормализации"""
    return text.lower().strip('«»"')

# Золотой стандарт (что ДОЛЖНА найти идеальная модель)
true_entities = {
    normalize('Денис Крючков'),
    normalize('Москва'),
    normalize('Хабр'),
    normalize('Mail.Ru Group'),
    normalize('Habr Blockchain Publishing LTD'),
    normalize('Никосия'),
    normalize('Европа'),
}

# Нормализуем результат модели
found_entities = {normalize(span.text) for span in doc_n.spans}

tp = len(found_entities & true_entities)
fp = len(found_entities - true_entities)
fn = len(true_entities - found_entities)

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Эталон (золотой стандарт): {true_entities}")
print(f"\nНайдено моделью: {found_entities}")
print(f"\nTP={tp}, FP={fp}, FN={fn}")
print(f"Precision: {precision:.2f}  |  Recall: {recall:.2f}  |  F1: {f1:.2f}")

---
<a id='part4'></a>
# Часть 4. RegEx — «Когда правила сильнее нейросетей»
## Гибридный извлекатель: Natasha + RegEx

In [ ]:
import re

text_hybrid = """
«Хабр» — крупнейшее IT-сообщество, основанное Денисом Крючковым.
Контракт №45-912/А был подписан 20.10.2023. Для связи используйте email:
info@habr.com или support@habr.ru. Юридический адрес: 123456, г. Москва,
ул. Ленина, д. 5, кв. 10. ИНН организации: 7712345678.
Телефон: +7 (495) 123-45-67. СНИЛС: 123-456-789 00.
"""

# 1. Natasha: находим организации и людей
doc_h = Doc(text_hybrid)
doc_h.segment(segmenter)
doc_h.tag_ner(ner_tagger)
orgs = [s.text for s in doc_h.spans if s.type == 'ORG']
pers = [s.text for s in doc_h.spans if s.type == 'PER']
print(f"Natasha — Организации: {orgs}")
print(f"Natasha — Люди:        {pers}")
print()

# 2. RegEx: находим то, что нейросеть не «видит»
patterns = {
    'ИНН':       r'\b\d{10}\b',
    'Email':     r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
    'Контракт':  r'№\d{2}-\d{3}/[А-Я]',
    'Телефон':   r'\+?[78][\s\-]?\(?\d{3}\)?[\s\-]?\d{3}[\s\-]?\d{2}[\s\-]?\d{2}',
    'Дата':      r'\d{2}\.\d{2}\.\d{4}',
    'Индекс':    r'\b\d{6}\b',
}

for name, pattern in patterns.items():
    found = re.findall(pattern, text_hybrid)
    if found:
        print(f"RegEx [{name:10s}]: {found}")

---
<a id='part5'></a>
# Часть 5. Векторные представления слов
## Word2Vec, FastText (Gensim) + Navec визуализация

In [ ]:
# Датасет для обучения Word2Vec / FastText
from gensim.models import Word2Vec, FastText
from gensim.utils import simple_preprocess
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

raw_data = [
    "Хабр — популярная платформа для IT специалистов и разработчиков",
    "На Хабре можно найти полезные статьи по программированию и технологиям",
    "Пользователи Хабра делятся своим уникальным опытом и знаниями",
    "Программист пишет чистый код на языке программирования Питон",
    "Разработчик создает современные мобильные приложения для смартфонов",
    "Системный администратор настраивает защищенный сервер и локальную сеть",
    "Программирование — это искусство написания эффективного кода",
    "Опытные специалисты обсуждают новые технологии в комментариях на Хабре",
    "Машинное обучение и нейронные сети изменяют мир технологий",
    "Искусственный интеллект помогает автоматизировать рутинные задачи",
]

sentences = [simple_preprocess(text) for text in raw_data]
print(f"Датасет готов: {len(sentences)} предложений")
print(f"Пример: {sentences[0]}")

In [ ]:
# Word2Vec
w2v = Word2Vec(sentences=sentences, vector_size=50, window=3, min_count=1, workers=4, sg=0)
print("Word2Vec обучен ✓")
print(f"Размер словаря: {len(w2v.wv)}")
print(f"\nВектор 'хабр' (первые 10): {w2v.wv['хабр'][:10]}")
print("\nПохожие слова для 'хабр' (Word2Vec):")
for word, sim in w2v.wv.most_similar('хабр', topn=5):
    print(f"  {word:20s} {sim:.3f}")

In [ ]:
# FastText — учитывает морфологию (n-граммы)
ft = FastText(vector_size=50, window=3, min_count=1, workers=4)
ft.build_vocab(corpus_iterable=sentences)
ft.train(corpus_iterable=sentences, total_examples=len(sentences), epochs=10)

print("FastText обучен ✓")
print("\nПохожие слова для 'хабр' (FastText):")
for word, sim in ft.wv.most_similar('хабр', topn=5):
    print(f"  {word:20s} {sim:.3f}")

# FastText работает даже с OOV словами
try:
    vec = ft.wv['хабронавт']  # Несуществующее слово!
    print(f"\nFastText нашел вектор для 'хабронавт' (OOV): {vec[:5]}...")
except KeyError:
    print("\nWord2Vec не может найти OOV слово")

In [ ]:
# Тест: опечатки
print("Тест на опечатки:")
for typo in ['програмист', 'разработчикк', 'хабрр']:
    try:
        sim_w = w2v.wv.most_similar(typo, topn=1)
        print(f"  Word2Vec [{typo}]: {sim_w[0]}")
    except KeyError:
        print(f"  Word2Vec [{typo}]: ❌ слово не в словаре")
    try:
        sim_f = ft.wv.most_similar(typo, topn=1)
        print(f"  FastText  [{typo}]: {sim_f[0]}")
    except KeyError:
        print(f"  FastText  [{typo}]: ❌")

In [ ]:
# Семантическая арифметика через предобученную GloVe (английский)
import gensim.downloader as api

print("Загружаем GloVe glove-wiki-gigaword-100...")
glove_model = api.load("glove-wiki-gigaword-100")

# Лишнее слово
words_test = ["breakfast", "cereal", "dinner", "lunch", "bicycle"]
extra = glove_model.doesnt_match(words_test)
print(f"\nЛишнее в {words_test}: '{extra}'")

# King - Man + Woman = ?
result = glove_model.most_similar(positive=['woman', 'king'], negative=['man'], topn=1)
print(f"King − Man + Woman = '{result[0][0]}'")

# Paris : France = ? : Germany
cap = glove_model.most_similar(positive=['germany', 'paris'], negative=['france'], topn=1)
print(f"Столица Германии (France:Paris = Germany:?): '{cap[0][0]}'")

In [ ]:
# Визуализация семантических кластеров (PCA)
def plot_words(model, words, title='Карта смыслов слов'):
    vectors = []
    valid_words = []
    for w in words:
        try:
            vectors.append(model.wv[w])
            valid_words.append(w)
        except KeyError:
            pass
    if len(vectors) < 3:
        print("Недостаточно слов для PCA")
        return
    coords = PCA(n_components=2).fit_transform(vectors)
    plt.figure(figsize=(10, 7))
    plt.scatter(coords[:, 0], coords[:, 1], c='#e74c3c', s=80, zorder=3)
    for i, word in enumerate(valid_words):
        plt.annotate(word, xy=(coords[i, 0], coords[i, 1]),
                     xytext=(6, 3), textcoords='offset points', fontsize=11)
    plt.title(title, fontsize=14)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

test_words = ["хабр", "статьи", "технологиям", "программист", "разработчик",
              "код", "питон", "сеть", "сервер", "платформа"]
plot_words(ft, test_words, 'FastText — карта смыслов')

---
<a id='part6'></a>
# Часть 6. Нейросети для NLP
## 6.1 Keras SimpleRNN и 6.2 PyTorch LSTM

In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

# Датасет: отзывы о Хабре
texts  = [
    "Хабр — популярная платформа для IT-специалистов",
    "На Хабре можно найти статьи по программированию и технологиям",
    "Пользователи Хабра делятся своим опытом и знаниями",
    "Хабр помогает разработчикам узнавать о новых технологиях",
    "Хабр — отличный ресурс для IT-специалистов",
    "Я обожаю Хабр за полезные статьи и отличные комментарии",
    "Не все статьи на Хабре полезны и понятны",
    "Иногда на Хабре сложно найти нужную информацию",
    "Хабр может быть сложен для понимания новичками",
    "Некоторые статьи на Хабре слишком технически сложны",
]
labels = np.array([1, 1, 1, 1, 1, 1, 0, 0, 0, 0], dtype='float32')

maxlen = 12
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)
seqs = tokenizer.texts_to_sequences(texts)
data = pad_sequences(seqs, maxlen=maxlen)

word_index = tokenizer.word_index
print(f"Словарь: {len(word_index)} слов")

In [ ]:
# Keras SimpleRNN
model_rnn = Sequential([
    Embedding(input_dim=len(word_index) + 1, output_dim=32),
    SimpleRNN(32),
    Dense(1, activation='sigmoid'),
])
model_rnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_rnn.summary()

history_rnn = model_rnn.fit(data, labels, epochs=30, verbose=0)
final_acc = history_rnn.history['accuracy'][-1]
print(f"\nТочность после 30 эпох: {final_acc*100:.1f}%")

# График потерь
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_rnn.history['loss'], color='#e74c3c', linewidth=2)
plt.title('Потери SimpleRNN'); plt.xlabel('Эпоха'); plt.grid(alpha=0.3)
plt.subplot(1, 2, 2)
plt.plot(history_rnn.history['accuracy'], color='#2ecc71', linewidth=2)
plt.title('Точность SimpleRNN'); plt.xlabel('Эпоха'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Тест на новых данных (Keras)
test_texts_rnn = [
    "Я нашел полезную статью на Хабре о машинном обучении",
    "На Хабре слишком много сложной информации",
    "Хабр — отличный ресурс для разработчиков",
    "Сложно разобраться в статьях без подготовки",
]

test_seqs = tokenizer.texts_to_sequences(test_texts_rnn)
test_data = pad_sequences(test_seqs, maxlen=maxlen)
preds = model_rnn.predict(test_data, verbose=0)

print("Предсказания SimpleRNN:")
for text, pred in zip(test_texts_rnn, preds):
    label = '👍 Позитивный' if pred[0] >= 0.5 else '👎 Негативный'
    print(f"  [{pred[0]:.3f}] {label}")
    print(f"  '{text}'\n")

In [ ]:
# PyTorch LSTM
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, maxlen):
        self.data = []
        for text, label in zip(texts, labels):
            seq = tokenizer.texts_to_sequences([text])[0]
            if len(seq) < maxlen:
                seq = np.pad(seq, (0, maxlen - len(seq)), 'constant')
            else:
                seq = seq[:maxlen]
            self.data.append((
                torch.tensor(seq, dtype=torch.long),
                torch.tensor(label, dtype=torch.float32)
            ))
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

class LSTMModel(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.lstm      = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.linear    = nn.Linear(hidden_dim, 1)
        self.sigmoid   = nn.Sigmoid()
    def forward(self, x):
        emb = self.embedding(x)
        _, (hn, _) = self.lstm(emb)
        return self.sigmoid(self.linear(hn[-1]))

labels_np = np.array([1, 1, 1, 1, 1, 1, 0, 0, 0, 0], dtype='float32')
dataset    = TextDataset(texts, labels_np, tokenizer, maxlen)
loader     = DataLoader(dataset, batch_size=2, shuffle=True)

lstm_model = LSTMModel(len(word_index) + 1, 32, 32)
criterion  = nn.BCELoss()
optimizer  = torch.optim.Adam(lstm_model.parameters(), lr=0.005)

lstm_model.train()
losses_pt = []
for epoch in range(30):
    total = 0
    for seqs_b, labels_b in loader:
        optimizer.zero_grad()
        out  = lstm_model(seqs_b).squeeze()
        loss = criterion(out, labels_b)
        loss.backward()
        optimizer.step()
        total += loss.item()
    losses_pt.append(total / len(loader))
    if (epoch + 1) % 10 == 0:
        print(f"  Эпоха {epoch+1:3d}  |  Потери: {losses_pt[-1]:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(losses_pt, color='#9b59b6', linewidth=2)
plt.title('Потери PyTorch LSTM'); plt.xlabel('Эпоха'); plt.ylabel('Loss')
plt.grid(alpha=0.3); plt.show()

In [ ]:
# Тест на новых данных (PyTorch)
test_seqs_pt = []
for text in test_texts_rnn:
    seq = tokenizer.texts_to_sequences([text])[0]
    seq = np.pad(seq, (0, max(0, maxlen - len(seq))), 'constant')[:maxlen]
    test_seqs_pt.append(seq)

test_tensor = torch.tensor(test_seqs_pt, dtype=torch.long)
lstm_model.eval()
with torch.no_grad():
    preds_pt = lstm_model(test_tensor).squeeze().numpy()

print("Предсказания PyTorch LSTM:")
for text, pred in zip(test_texts_rnn, preds_pt):
    label = '👍 Позитивный' if pred >= 0.5 else '👎 Негативный'
    print(f"  [{pred:.3f}] {label}: '{text[:50]}...'") 

---
<a id='part7'></a>
# Часть 7. Классификация текстов
## scikit-learn (20 Newsgroups) + Lenta.ru с Natasha

In [ ]:
# 7.1 Классификация 20 Newsgroups (англ.)
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

categories = ['sci.space', 'rec.sport.baseball', 'soc.religion.christian', 'talk.politics.guns']
newsgroups = fetch_20newsgroups(subset='all', categories=categories, remove=('headers', 'footers', 'quotes'))

X_train, X_test, y_train, y_test = train_test_split(
    newsgroups.data, newsgroups.target, test_size=0.2, random_state=42)

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words='english')
X_train_tf = tfidf.fit_transform(X_train)
X_test_tf  = tfidf.transform(X_test)

# Наивный Байес
nb = MultinomialNB()
nb.fit(X_train_tf, y_train)
y_pred_nb = nb.predict(X_test_tf)

# Логистическая регрессия
lr = LogisticRegression(max_iter=500, solver='lbfgs')
lr.fit(X_train_tf, y_train)
y_pred_lr = lr.predict(X_test_tf)

print(f"Naive Bayes accuracy:       {accuracy_score(y_test, y_pred_nb):.3f}")
print(f"Logistic Regression accuracy: {accuracy_score(y_test, y_pred_lr):.3f}\n")
print("Детальный отчет (Logistic Regression):")
print(classification_report(y_test, y_pred_lr, target_names=newsgroups.target_names))

In [ ]:
# Матрица ошибок
cm = confusion_matrix(y_test, y_pred_lr)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=[c.split('.')[-1] for c in newsgroups.target_names],
            yticklabels=[c.split('.')[-1] for c in newsgroups.target_names],
            cmap='Blues')
plt.ylabel('Истинный класс')
plt.xlabel('Предсказанный класс')
plt.title('Матрица ошибок (Logistic Regression)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Функция предсказания темы по тексту
def predict_news_en(text):
    vec = tfidf.transform([text])
    pred = lr.predict(vec)[0]
    proba = lr.predict_proba(vec).max()
    return f"{newsgroups.target_names[pred]} ({proba:.0%})"

tests = [
    "The Hubble telescope captured amazing images of distant galaxies.",
    "The pitcher threw a perfect curveball in the ninth inning.",
    "God bless everyone who reads this prayer.",
    "The government wants to ban assault weapons.",
]
for t in tests:
    print(f"  {predict_news_en(t):50s} | {t[:60]}")

In [ ]:
# 7.2 Lenta.ru — русскоязычные новости с Natasha
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

print("Загрузка датасета Lenta.ru...")
data = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "yutkin/corpus-of-russian-news-articles-from-lenta",
    "lenta-ru-news.csv",
    pandas_kwargs={"nrows": 10000}
)

# Топ-5 тем, по 100 примеров каждой
top_topics = data['topic'].value_counts().head(5).index.tolist()
data = data[data['topic'].isin(top_topics)].dropna(subset=['title', 'topic'])
df = data.groupby('topic', group_keys=False).apply(lambda x: x.sample(100, random_state=42)).reset_index(drop=True)

print(f"Датасет загружен: {df.shape[0]} строк")
print(df['topic'].value_counts().to_string())

In [ ]:
# Очистка через Natasha (лемматизация существительных и прилагательных)
def clean_ru(text):
    if not text or not isinstance(text, str): return ""
    doc_tmp = Doc(text)
    doc_tmp.segment(segmenter)
    doc_tmp.tag_morph(morph_tagger)
    lemmas = []
    for t in doc_tmp.tokens:
        t.lemmatize(morph_vocab)
        if t.lemma and t.pos in ['NOUN', 'ADJ'] and len(t.lemma) > 2:
            lemmas.append(t.lemma.lower())
    return " ".join(lemmas)

print("Лемматизация заголовков (это займёт ~1-2 мин)...")
df['cleaned'] = df['title'].apply(clean_ru)
print("✓ Готово")
print("Примеры:")
for _, row in df.head(3).iterrows():
    print(f"  [{row['topic']:15s}] '{row['title'][:60]}'")
    print(f"  {'':17s} → '{row['cleaned'][:60]}'")

In [ ]:
# Обучение и сравнение Naive Bayes vs Logistic Regression на русском
X_tr, X_te, y_tr, y_te = train_test_split(
    df['cleaned'], df['topic'], test_size=0.2, random_state=42, stratify=df['topic'])

vec_ru = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_tr_tf = vec_ru.fit_transform(X_tr)
X_te_tf = vec_ru.transform(X_te)

nb_ru = MultinomialNB(alpha=0.1)
nb_ru.fit(X_tr_tf, y_tr)

lr_ru = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_ru.fit(X_tr_tf, y_tr)

acc_nb = accuracy_score(y_te, nb_ru.predict(X_te_tf))
acc_lr = accuracy_score(y_te, lr_ru.predict(X_te_tf))

print(f"Naive Bayes:           {acc_nb:.3f}")
print(f"Logistic Regression:   {acc_lr:.3f}")
print()
print(classification_report(y_te, lr_ru.predict(X_te_tf)))

In [ ]:
# Интерактивный предсказатель темы
def predict_ru_news(title):
    cleaned = clean_ru(title)
    vec = vec_ru.transform([cleaned])
    pred = lr_ru.predict(vec)[0]
    proba = lr_ru.predict_proba(vec).max()
    return f"{pred} ({proba:.0%})"

headlines = [
    "Сборная России по футболу одержала победу",
    "Курс доллара установил новый рекорд на бирже",
    "Ученые открыли новый химический элемент",
    "Яндекс запустил новый сервис для разработчиков",
]
for h in headlines:
    print(f"  {predict_ru_news(h):30s} | {h}")

In [ ]:
# Топ-слова для каждой темы
feature_names = vec_ru.get_feature_names_out()
print("Топ-10 характерных слов для каждой темы (Bayesian log-proba):")
for i, cls in enumerate(nb_ru.classes_):
    top_idx = np.argsort(nb_ru.feature_log_prob_[i])[-10:]
    words_top = [feature_names[j] for j in top_idx]
    print(f"  ** {cls:<20} **: {', '.join(words_top)}")

---
<a id='part8'></a>
# Часть 8. BERT — трансформер для NLP
## Анализ тональности русских отзывов (Hugging Face)

In [ ]:
# 8.1 Zero-shot: используем готовую модель без дообучения
from transformers import pipeline

print("Загружаем модель (это займёт ~1 минуту)...")
# Используем multilingual-модель для анализа тональности
sentiment_pipeline = pipeline(
    'sentiment-analysis',
    model='nlptown/bert-base-multilingual-uncased-sentiment'
)

reviews = [
    "Хабр — отличная платформа! Очень полезные статьи.",
    "Ужасный сайт, ничего не понятно, очень сложно.",
    "Неплохой ресурс, но иногда бывает сложновато.",
    "Обожаю Хабр! Лучший IT-ресурс на русском языке.",
    "Скучно и бесполезно. Трата времени.",
]

print("\nАнализ тональности (BERT multilingual):")
print(f"{'Отзыв':<50} {'Оценка'}") 
print("-" * 70)
for review in reviews:
    result = sentiment_pipeline(review[:512])[0]
    stars = result['label']  # 1-5 звёзд
    score = result['score']
    print(f"{review[:48]:<50} {stars} ({score:.2f})")

In [ ]:
# 8.2 Дообучение BERT на своём датасете (fine-tuning)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import Dataset
import torch

# Датасет для fine-tuning
train_texts_bert = [
    "Хабр — отличный ресурс", "На Хабре много полезного", "Обожаю Хабр!",
    "Хабр помогает развиваться", "Отличные статьи на Хабре", "Очень полезная платформа",
    "Хабр непонятен новичкам", "Слишком сложно", "Трудно разобраться",
    "Не нравится интерфейс", "Много ненужной информации", "Скучно и однообразно",
]
train_labels_bert = [1,1,1,1,1,1, 0,0,0,0,0,0]

MODEL_NAME = 'DeepPavlov/rubert-base-cased'
print(f"Загружаем токенизатор {MODEL_NAME}...")
tokenizer_bert = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer_bert(examples['text'], padding='max_length',
                          truncation=True, max_length=64)

train_ds = Dataset.from_dict({'text': train_texts_bert, 'label': train_labels_bert})
train_ds = train_ds.map(tokenize_fn, batched=True)

print(f"Загружаем модель {MODEL_NAME}...")
model_bert = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir='./bert_habr',
    num_train_epochs=5,
    per_device_train_batch_size=4,
    logging_steps=10,
    save_strategy='no',
    no_cuda=False,
    report_to='none',
)

trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=train_ds,
)

print("\nДообучение BERT...")
trainer.train()
print("\nГотово! ✓")

In [ ]:
# Предсказания с дообученным BERT
def predict_bert(texts_list, model, tokenizer, device='cpu'):
    model.eval()
    inputs = tokenizer(texts_list, padding=True, truncation=True,
                       max_length=64, return_tensors='pt')
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)
    preds = probs.argmax(dim=1).numpy()
    confs = probs.max(dim=1).values.numpy()
    return preds, confs

test_bert = [
    "Хабр — великолепная платформа для профессионалов",
    "Невозможно разобраться, всё слишком запутано",
    "Хорошие статьи, но иногда занудно",
    "Лучший ресурс для IT-специалистов!",
]

preds_bert, confs_bert = predict_bert(test_bert, model_bert, tokenizer_bert)

print("Предсказания дообученного BERT:")
for text, pred, conf in zip(test_bert, preds_bert, confs_bert):
    label = '👍 Позитивный' if pred == 1 else '👎 Негативный'
    print(f"  [{conf:.2f}] {label}")
    print(f"  '{text}'\n")

---
## 📊 Итоговая сводка

In [ ]:
print("=" * 70)
print("ИТОГОВАЯ ТАБЛИЦА: Инструменты NLP")
print("=" * 70)
rows_summary = [
    ("NLTK",           "Токенизация, стемминг",        "Любой язык",     "Обучение"),
    ("pymorphy3",      "Лемматизация",                 "Русский",        "Морфология"),
    ("spaCy",          "POS, NER, dependency",          "Много языков",   "Продакшн"),
    ("Natasha",        "NER, POS, адреса, ФИО",        "Только RU",      "Русский спец."),
    ("RegEx",          "ИНН, email, телефоны",          "Любой",         "Паттерны"),
    ("Word2Vec",       "Семантика слов",                "Любой",         "Эмбеддинги"),
    ("FastText",       "Семантика + морфология",        "Лучший для RU",  "OOV слова"),
    ("Keras SimpleRNN","Классификация текста",          "Любой",         "Простая RNN"),
    ("PyTorch LSTM",   "Классификация текста",          "Любой",         "Гибкость"),
    ("Naive Bayes",    "Тематика, спам",                "Любой",         "Скорость"),
    ("Log. Regression","Тематика (лучше NB)",           "Любой",         "Точность"),
    ("BERT",           "Всё: тональность, NER, QA",     "Много языков",  "SOTA"),
]
print(f"  {'Инструмент':<20} {'Задача':<28} {'Язык':<15} Когда")
print("  " + "-" * 80)
for row in rows_summary:
    print(f"  {row[0]:<20} {row[1]:<28} {row[2]:<15} {row[3]}")
print("=" * 70)

---
## 🎯 Итоговый проект — заготовка: «Яндекс в зеркале NLP»

In [ ]:
# Этап 1: Очистка и фильтрация (NLTK + RegEx)
# Этап 2: NER через Natasha (персоны, орги, локации)
# Этап 3: Синтаксический анализ spaCy
# Этап 4: Семантический поиск (Word2Vec / FastText)
# Этап 5: Классификация тональности (LSTM / BERT)

SAMPLE_NEWS = [
    "Яндекс объявил о запуске нового поискового алгоритма в 2026 году.",
    "Генеральный директор Яндекса встретился с представителями Google в Москве.",
    "Яндекс.Маркет расширяет сеть доставки в регионах России.",
    "Акции Яндекса выросли на 15% после публикации квартального отчёта.",
    "Критики считают, что алгоритм Яндекса продвигает только платный контент.",
]

print("=== Этап 1: Предобработка ===")
for text in SAMPLE_NEWS:
    clean = preprocess_text(text)
    print(f"  {text[:60]}...\n  → {clean[:10]}\n")

print("\n=== Этап 2: NER (Natasha) ===")
for text in SAMPLE_NEWS:
    doc_tmp = Doc(text)
    doc_tmp.segment(segmenter)
    doc_tmp.tag_ner(ner_tagger)
    entities = [(s.text, s.type) for s in doc_tmp.spans]
    if entities:
        print(f"  {text[:50]}...")
        for ent, typ in entities:
            print(f"    [{typ}] {ent}")

---
*Практикум NLP · «Когда машины начинают понимать нас» · 2026*